# inflow-ledger: what the archive holds

Observed daily reservoir inflow (CDEC) with the newest archived HEFS ensemble as a fan, then the score table.
Run `python archive.py && python score.py` from the repo root first, or check out the `data` branch into `data/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from score import LEADS, SCORED, read_hefs, read_obs

DATA = Path.cwd().parent / "data"


In [ ]:
def latest_fan(lid):
    """Percentiles per forecast day from the newest archived ensemble, in cfs."""
    path = sorted((DATA / "hefs_daily" / lid).glob("*.gz"))[-1]
    fc = read_hefs(path)
    q = {d: np.percentile(m, [5, 25, 50, 75, 95]) for d, m in fc.items()}
    return pd.DataFrame(q, index=["p05", "p25", "p50", "p75", "p95"]).T.iloc[: LEADS.stop], path.stem

fig, axes = plt.subplots(4, 3, figsize=(15, 13), sharex=False)
for ax, (lid, sta) in zip(axes.flat, SCORED.items()):
    obs = pd.Series(read_obs(DATA / "cdec_inflow" / f"{sta}.csv")).sort_index()
    fan, issued = latest_fan(lid)
    recent = obs[obs.index >= fan.index[0] - pd.Timedelta(days=60)]
    ax.plot(recent.index, recent.values, "k.-", lw=1, ms=3, label="CDEC observed")
    ax.fill_between(fan.index, fan.p05, fan.p95, color="C0", alpha=0.2, label="HEFS 5-95%")
    ax.fill_between(fan.index, fan.p25, fan.p75, color="C0", alpha=0.4, label="HEFS 25-75%")
    ax.plot(fan.index, fan.p50, color="C0", label="HEFS median")
    ax.set_title(f"{lid} / {sta}  (issued {issued[:8]})", fontsize=10)
    ax.set_ylabel("cfs")
    ax.tick_params(axis="x", labelrotation=30, labelsize=8)
axes.flat[0].legend(fontsize=8)
for ax in axes.flat[len(SCORED):]:
    ax.axis("off")
fig.tight_layout()


In [ ]:
scores = pd.read_csv(DATA / "scores" / "inflow_daily.csv", parse_dates=["issue"])
print(f"{len(scores)} scored forecast-days, {scores.issue.nunique()} issue days, {scores.lid.nunique()} points")
if len(scores):
    by_lead = scores.groupby("lead")[["crps_hefs", "crps_persistence", "crps_climatology", "in90_hefs"]].mean()
    display(by_lead.round(1))
    ax = by_lead[["crps_hefs", "crps_persistence", "crps_climatology"]].plot(marker="o", figsize=(8, 4))
    ax.set(xlabel="lead (days)", ylabel="mean CRPS (cfs)", title="Pooled over scored points")
